# 05.8 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/05-powerful-models/08-explaining-models/exercises.ipynb)

Exercises 1 and 2 are on paper. Check them with the code cells after you have an answer.

## Exercise 1: Shapley by hand (paper)

A two-feature model: base 20, A alone 20, B alone 50, both 60. Compute each feature's Shapley value. What is odd about A?

*Your answer here.*

## Exercise 2: permutation by hand (paper)

Use the six-row table from the README. Shuffle `days_since_last` with the order `[3, 4, 5, 0, 1, 2]` (i.e. the first three and last three swap). What is the accuracy, and the importance?

*Your answer here.*

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

import lightgbm as lgb
from sklearn.metrics import roc_auc_score
# the tuned settings found by grid search in 05.6
model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.01, num_leaves=4,
                           verbose=-1, random_state=42, n_jobs=1).fit(X_train, y_train)
print(f"tuned LightGBM, test AUC {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.3f}")

six = pd.DataFrame({"orders": [5, 2, 8, 1, 3, 1], "days_since_last": [10, 20, 30, 90, 100, 120],
                    "churned": [0, 0, 0, 1, 1, 1]})
rule = lambda df: (df["days_since_last"] > 60).astype(int)     # the "model"
acc = lambda df: (rule(df) == df["churned"]).mean()

from itertools import permutations
from math import factorial

def shapley(value, players):
    """Average each player's marginal contribution over every joining order."""
    credit = {p: 0.0 for p in players}
    for order in permutations(players):
        joined = frozenset()
        for p in order:
            credit[p] += value[joined | {p}] - value[joined]
            joined = joined | {p}
    return {p: c / factorial(len(players)) for p, c in credit.items()}

## Exercise 3: importance depends on the metric

Compute permutation importance for the tuned LightGBM with `scoring="accuracy"` and with `scoring="roc_auc"`. Does the ranking of features change?

In [ ]:
# your code here

## Exercise 4: explain a loyal customer

Find the test customer with the **lowest** predicted churn risk. Draw their waterfall and write a two-sentence explanation for a retention manager.

In [ ]:
# your code here